# Cross-Cohort Integration of Epicenters (UnifyEpicenter)

Merge the individual-level epicenter frequency maps from ABIDE-II and CABIC, and define **core epicenters** via rank-based integration of the two cohort-specific rankings. The final integrated score is the average rank across the two cohorts.

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================================================
# Configuration
# ============================================================
ABIDE2_FREQ_CSV = 'output/Center/ASD_Total_Best_Seeds_MinGOF_aparc.csv'
CABIC_FREQ_CSV = 'output/Center/CABIC_ASD_Total_Best_Seeds_MinGOF_aparc.csv'
OUTPUT_DIR = 'output/UnifyEpicenter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------
# Load cohort-specific frequency maps
# ----------------------------------------------------------
df_abide2 = pd.read_csv(ABIDE2_FREQ_CSV, index_col='Brain_Region')
df_cabic = pd.read_csv(CABIC_FREQ_CSV, index_col='Brain_Region')

# Rename columns with cohort suffix for clarity
df_abide2.columns = [f'{col}_ABIDE2' for col in df_abide2.columns]
df_cabic.columns = [f'{col}_CABIC' for col in df_cabic.columns]

# Merge on brain region
merged_df = df_abide2.join(df_cabic, how='outer').fillna(0)

# ----------------------------------------------------------
# Rank-based integration
# ----------------------------------------------------------
# Rank each cohort's count in descending order (1 = most frequent)
merged_df['Rank_ABIDE2'] = merged_df['Count_Total_ASD_ABIDE2'].rank(ascending=False, method='average')
merged_df['Rank_CABIC'] = merged_df['Count_Total_ASD_CABIC'].rank(ascending=False, method='average')

# Integrated score = average of the two ranks (normalized to total N)
n_regions = len(merged_df)
merged_df['Integrated_Score'] = (merged_df['Rank_ABIDE2'] + merged_df['Rank_CABIC']) / n_regions

# Sort by integrated score (most core epicenter first)
merged_df = merged_df.sort_values('Integrated_Score', ascending=True)

# ----------------------------------------------------------
# Format and save
# ----------------------------------------------------------
# Keep informative columns and round the integrated score to 4 decimals
summary_df = merged_df.reset_index()
summary_df = summary_df[['Brain_Region',
                          'Count_Total_ASD_ABIDE2', 'Frequency_Total_ASD_ABIDE2',
                          'Count_Total_ASD_CABIC', 'Frequency_Total_ASD_CABIC',
                          'Rank_ABIDE2', 'Rank_CABIC', 'Integrated_Score']]
summary_df.columns = ['Brain Region',
                      'Count (ABIDE-2)', 'Frequency (ABIDE-2, %)',
                      'Count (CABIC)', 'Frequency (CABIC, %)',
                      'Rank ABIDE-2', 'Rank CABIC', 'Integrated Score']
summary_df['Frequency (ABIDE-2, %)'] = (summary_df['Frequency (ABIDE-2, %)'] * 100).round(2)
summary_df['Frequency (CABIC, %)'] = (summary_df['Frequency (CABIC, %)'] * 100).round(2)
summary_df['Integrated Score'] = summary_df['Integrated Score'].round(4)

out_path = os.path.join(OUTPUT_DIR, 'Table_S1_Epicenter_Comparison.xlsx')
summary_df.to_excel(out_path, index=False)
print(f"Saved to {out_path}")
print()
print("Top 12 core epicenters across cohorts:")
print(summary_df.head(12).to_string(index=False))